# Reference only: do not run

Earlier versions of the Bronze load, kept for learning:

* **v1 (KAN-10):** one cell per table, copy-pasted
* **v2 (KAN-11, first try):** `table_create`, which scans the folder but uses inferSchema for every file

Superseded by `ingest_csv` in `notebooks/01_bronze_ingest` (KAN-11). Running this notebook would overwrite the Bronze tables without audit columns.

### lets create dataframe for these orders, customers, order_items, products, payments 

## Orders Table

In [0]:
from pyspark.sql.types import *

schema = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True)])

orders = spark.read.format("csv").option("header", True).schema(schema).load("/Volumes/shopkart/bronze/raw_files/olist/olist_orders_dataset.csv")
orders.write.mode("overwrite").saveAsTable("shopkart.bronze.orders")
orders.count()

In [0]:
orders.schema

## Customers Table

In [0]:
import pyspark.sql.functions as F

customers = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/shopkart/bronze/raw_files/olist/olist_customers_dataset.csv")
display(customers)

customers = customers.withColumns({
    "_ingested_at": F.current_timestamp(),
    "_source_file":F.lit("/Volumes/shopkart/bronze/raw_files/olist/olist_customers_dataset.csv")}) 

customers.show(5)
#customers.write.mode("overwrite").saveAsTable("shopkart.bronze.customers")
customers.count()

## order_items

In [0]:
order_items = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/shopkart/bronze/raw_files/olist/olist_order_items_dataset.csv")
display(order_items)
order_items.write.mode("overwrite").saveAsTable("shopkart.bronze.order_items")
order_items.count()

### products


In [0]:
products = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/shopkart/bronze/raw_files/olist/olist_products_dataset.csv")
display(products)
products.write.mode("overwrite").saveAsTable("shopkart.bronze.products")
products.count()

### Payments


In [0]:
payments = spark.read.format("csv").option("header", True).option("inferSchema", True).load("/Volumes/shopkart/bronze/raw_files/olist/olist_order_payments_dataset.csv")
display(payments)
payments.write.mode("overwrite").saveAsTable("shopkart.bronze.payments")
payments.count()

## v2: `table_create` (folder scan, inferSchema only)

In [0]:


def table_create(volume_path, catalog_schema):
    """Read all CSV files from a volume directory and write them as Delta tables.

    Args:
        volume_path: UC volume path to scan for CSV files
        catalog_schema: Target catalog.schema for the output tables
    """
    file_names = [file.name for file in dbutils.fs.ls(volume_path) if file.name.endswith(".csv")]
    print(f"Found {len(file_names)} CSV files: {file_names}")

    for name in file_names:
        table_name = name.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
        df = spark.read.format("csv").option("header", True).option("inferSchema", True).load(f"{volume_path}/{name}")
        df = df.withColumns({
            "_ingested_at": F.current_timestamp(),
            "_source_file": F.col("_metadata.file_path")
        })
        df.write.mode("overwrite").saveAsTable(f"{catalog_schema}.{table_name}")
        print(f"Created table: {catalog_schema}.{table_name} ({df.count()} rows)")


In [0]:
table_create("/Volumes/shopkart/bronze/raw_files/olist", "shopkart.bronze")

In [0]:
tables = ["orders", "customers", "order_items", "products", "order_payments"]

for t in tables:
    print(f"=== {t} ===")
    spark.table(f"shopkart.bronze.{t}") \
        .select("_source_file") \
        .distinct() \
        .show(truncate=False)